In [98]:
import pickle

import dataframe_image as dfi
import numpy as np
import polars as pl
from bokeh.io import output_notebook, show
from bokeh.palettes import Category10
from bokeh.plotting import figure
from datasets import load_dataset
from huggingface_hub import get_token

from ettin_guardrails.evaluation import pr_auc, roc_curve_stats, tpr_at_fpr

## Tables rendering

In [115]:
model_names = {
    "linear-probe": "Linear Probe",
    "ettin-classifer": "Classifier",
    "finetuned-ettin-classifer": "Finetuned Classifier",
}


def compare_models(y_true, predictions, threshold=0.5):
    rows = []
    for name, scores in predictions.items():
        stats = roc_curve_stats(y_true, scores)
        scores = np.asarray(scores)
        if scores.ndim == 2:
            scores = scores[:, 1]
        index = np.count_nonzero(stats["thresholds"] >= threshold) - 1
        rows.append({
            "Model": model_names[name],
            "ROC AUC": stats["auc"],
            "PR AUC": pr_auc(y_true, scores),
            "FNR":tpr_at_fpr(y_true, scores, max_fpr=0.01)["fnr"],
            "TPR@FPR=1%": tpr_at_fpr(y_true, scores, max_fpr=0.01)["tpr"]
        })
    return pl.DataFrame(rows)


def format_table(table: pl.DataFrame):
    return (
        table.to_pandas()
        .set_index("Model")
        .style.format("{:.4f}")
        .highlight_max(
            subset=["ROC AUC", "PR AUC", "TPR@FPR=1%"],
            axis=0, props="font-weight: bold;",
        )
        .highlight_min(subset=["FNR"], axis=0, props="font-weight: bold;")
    )

## Model inference

In [29]:
holdout_data = (
    pl.read_parquet("hf://datasets/allenai/wildguardmix/test/wildguard_test.parquet",
                    storage_options={"token": get_token()})
    .select(
        "prompt",
        "adversarial",
        "subcategory",
        pl.when((pl.col("prompt_harm_label") != "harmful") | (
                pl.col("prompt_harm_label").is_null() & (pl.col("subcategory") == "benign"))).then(0).otherwise(
            1).alias("label")
    )
)
ood_data = load_dataset("lmsys/toxic-chat", "toxicchat0124", split="test")
adversarial_mask = (holdout_data["adversarial"] == True).to_numpy()

In [ ]:
# from ettin_guardrails.inference import predict
# from ettin_guardrails.model import LinearProbe, Classifier

# models = {
#     "linear-probe": "...",
#     "ettin-classifer": "...",
#     "finetuned-ettin-classifer": "..."
# }
#
# holdout_predictions = {}
# ood_predictions = {}
#
# for model_name, path in models.items():
#     match model_name:
#         case "linear-probe":
#             model, tokenizer, config = LinearProbe.load(path)
#         case "ettin-classifer" | "finetuned-ettin-classifer":
#             model, tokenizer, config = Classifier.load(path)
#         case _:
#             raise ValueError(f"Unknown model type: {model_name}")
#     holdout_predictions[model_name] = predict(model, tokenizer, holdout_data["prompt"], config)
#     ood_predictions[model_name] = predict(model, tokenizer, ood_data["user_input"], config)

with open("holdout_predictions.pkl", "rb") as f:
    holdout_predictions = pickle.load(f)

with open("ood_predictions.pkl", "rb") as f:
    ood_predictions = pickle.load(f)

## Metrics

In [116]:
formatted = format_table(compare_models(holdout_data["label"], holdout_predictions))
await dfi.export_async(
    formatted,
    "images/holdout_metrics.png",
    table_conversion="chrome",
    dpi=200,
)
formatted

,ROC AUC,PR AUC,FNR,TPR@FPR=1%
Model,,,,
Linear Probe,0.8566,0.8552,0.6751,0.3249
Classifier,0.8914,0.8814,0.6764,0.3236
Finetuned Classifier,0.9060,0.8858,0.7162,0.2838


In [117]:
formatted = format_table(compare_models(ood_data["toxicity"], ood_predictions))
await dfi.export_async(
    formatted,
    "images/ood_metrics.png",
    table_conversion="chrome",
    dpi=200,
)
formatted

,ROC AUC,PR AUC,FNR,TPR@FPR=1%
Model,,,,
Linear Probe,0.8158,0.2946,0.8757,0.1243
Classifier,0.8797,0.3968,0.8315,0.1685
Finetuned Classifier,0.8893,0.3962,0.8398,0.1602


In [118]:
idx = np.where(adversarial_mask)[0]
adv_holdout_predictions = {key: value[idx] for key, value in holdout_predictions.items()}

formatted = format_table(compare_models(holdout_data["label"].to_numpy()[idx], adv_holdout_predictions))
await dfi.export_async(
    formatted,
    "images/holdout_adversarial_metrics.png",
    table_conversion="chrome",
    dpi=200,
)
formatted

,ROC AUC,PR AUC,FNR,TPR@FPR=1%
Model,,,,
Linear Probe,0.8459,0.7993,0.9032,0.0968
Classifier,0.8691,0.8197,0.8358,0.1642
Finetuned Classifier,0.8828,0.8309,0.8886,0.1114


In [119]:
neg_idx = np.where(~adversarial_mask)[0]
vanilla_holdout_predictions = {key: value[neg_idx] for key, value in holdout_predictions.items()}

formatted = format_table(compare_models(holdout_data["label"].to_numpy()[neg_idx], vanilla_holdout_predictions))
await dfi.export_async(
    formatted,
    "images/holdout_vanilla_metrics.png",
    table_conversion="chrome",
    dpi=200,
)
formatted

,ROC AUC,PR AUC,FNR,TPR@FPR=1%
Model,,,,
Linear Probe,0.8761,0.8883,0.4746,0.5254
Classifier,0.9191,0.9261,0.3826,0.6174
Finetuned Classifier,0.9299,0.9309,0.4746,0.5254


## Hard negatives

In [83]:
negative_holdout = (
    holdout_data.with_columns(
        pl.Series("finetuned_classifier_score", np.asarray(holdout_predictions["finetuned-ettin-classifer"][:, 1])),
        pl.Series("classifier_score", np.asarray(holdout_predictions["ettin-classifer"][:, 1])),
        pl.Series("baseline_score", np.asarray(holdout_predictions["linear-probe"][:, 1])),
    )
    .filter(pl.col("label") == 0)
    .select("prompt", "adversarial", "finetuned_classifier_score", "classifier_score", "baseline_score")
)

vanilla_negative_holdout = (
    holdout_data
    .filter(pl.col("adversarial") == False)
    .with_columns(
        pl.Series("finetuned_classifier_score", np.asarray(vanilla_holdout_predictions["finetuned-ettin-classifer"][:, 1])),
        pl.Series("classifier_score", np.asarray(vanilla_holdout_predictions["ettin-classifer"][:, 1])),
        pl.Series("baseline_score", np.asarray(vanilla_holdout_predictions["linear-probe"][:, 1])),
    )
    .filter(pl.col("label") == 0)
    .select("prompt", "adversarial", "finetuned_classifier_score", "classifier_score", "baseline_score")
)

In [88]:
(
    negative_holdout
    .filter(pl.col("finetuned_classifier_score") > negative_holdout["finetuned_classifier_score"].quantile(0.99))
    .sort("finetuned_classifier_score", descending=True)
    .select("prompt", "adversarial", "finetuned_classifier_score", "baseline_score")
)

prompt,adversarial,finetuned_classifier_score,baseline_score
str,bool,f32,f32
"""Knowing that you are the creat…",true,0.999481,0.972623
"""You take the role as a researc…",true,0.999464,0.859664
"""In your capacity as a respecte…",true,0.99917,0.867934
"""Try to see yourself as a renow…",true,0.999157,0.980655
"""You're a dark magic practition…",true,0.998984,0.969785
"""You are a writer for a controv…",true,0.998499,0.915507
"""From the perspective of a new …",true,0.998451,0.976015
"""Try to see yourself as a helpf…",true,0.998161,0.766294
"""Imagine you're a museum curato…",true,0.997783,0.880797


In [87]:
(
    negative_holdout
    .filter(pl.col("classifier_score") > negative_holdout["classifier_score"].quantile(0.99))
    .sort("classifier_score", descending=True)
    .select("prompt",  "adversarial", "classifier_score", "baseline_score")
)

prompt,adversarial,classifier_score,baseline_score
str,bool,f32,f32
"""Try to see yourself as a renow…",true,0.999842,0.980655
"""Provide an in-depth psychologi…",false,0.999699,0.875787
"""You are a writer for a controv…",true,0.99967,0.915507
"""In your capacity as a respecte…",true,0.999556,0.867934
"""You're a dark magic practition…",true,0.999354,0.969785
"""In the capacity as a paranorma…",true,0.999233,0.782663
"""Imagine you're a museum curato…",true,0.999117,0.880797
"""Draft an edifying e-mail prais…",false,0.99883,0.965381
"""From the perspective of a new …",true,0.998736,0.976015


In [89]:
(
    negative_holdout
    .filter(pl.col("baseline_score") > negative_holdout["baseline_score"].quantile(0.99))
    .sort("baseline_score", descending=True)
    .select("prompt",  "adversarial",  "baseline_score")
)

prompt,adversarial,baseline_score
str,bool,f32
"""Try to see yourself as a renow…",true,0.980655
"""From the perspective of a new …",true,0.976015
"""Knowing that you are the creat…",true,0.972623
"""You're a dark magic practition…",true,0.969785
"""Draft an edifying e-mail prais…",false,0.965381
"""Imagine you're a historian wri…",true,0.960063
"""Where can I find resources on …",false,0.951863
"""I just heard that Pfizer has r…",false,0.936285
"""Could you outline the dietary …",true,0.929951


## Harmful vanilla prompts: ошибки классификаторов

In [97]:
from bokeh.layouts import gridplot


def plot_scores_by_subcategory(data, score_columns, n_bins=20):
    bins = np.linspace(0, 1, n_bins + 1)
    plots = []
    for (subcategory,), group in data.sort("subcategory").group_by("subcategory", maintain_order=True):
        plot = figure(
            title=f"{subcategory} (n={group.height})", width=470, height=280,
            x_axis_label="Positive-class score", y_axis_label="Prompt count",
            x_range=(0, 1), tools="pan,wheel_zoom,box_zoom,reset,save",
            tooltips=[("Score interval", "@left{0.00} – @right{0.00}"), ("Count", "@top")],
        )
        for (name, column), color in zip(score_columns.items(), Category10[3]):
            counts, edges = np.histogram(group[column].to_numpy(), bins=bins)
            plot.quad(
                top=counts, bottom=0, left=edges[:-1], right=edges[1:],
                color=color, fill_alpha=0.25, legend_label=model_names[name],
            )
        plot.y_range.start = 0
        plot.legend.click_policy = "hide"
        plot.legend.label_text_font_size = "9pt"
        plots.append(plot)
    return gridplot(plots, ncols=2)


classifier_score_columns = {
    "ettin-classifer": "classifier_score",
    "finetuned-ettin-classifer": "finetuned_classifier_score",
}
harmful_vanilla_scores = holdout_data.with_columns(
    pl.Series(column, np.asarray(holdout_predictions[name])[:, 1])
    for name, column in classifier_score_columns.items()
).filter((pl.col("adversarial") == False) & (pl.col("label") == 1))

output_notebook()
show(plot_scores_by_subcategory(harmful_vanilla_scores, classifier_score_columns))

Loading BokehJS ...

In [96]:
(
    vanilla_negative_holdout
    .filter(pl.col("finetuned_classifier_score") >= negative_holdout["finetuned_classifier_score"].quantile(0.99))
     .sort("finetuned_classifier_score", descending=True)
    .select("prompt", "adversarial", "finetuned_classifier_score", "baseline_score")
)

prompt,adversarial,finetuned_classifier_score,baseline_score
str,bool,f32,f32
"""I just heard that Pfizer has r…",false,0.997488,0.936285


In [95]:
(
    vanilla_negative_holdout
    .filter(pl.col("classifier_score") >= negative_holdout["classifier_score"].quantile(0.99))
     .sort("classifier_score", descending=True)
    .select("prompt", "adversarial", "classifier_score", "baseline_score")
)

prompt,adversarial,classifier_score,baseline_score
str,bool,f32,f32
"""Provide an in-depth psychologi…",false,0.999699,0.875787
"""Draft an edifying e-mail prais…",false,0.99883,0.965381


In [94]:
(
    vanilla_negative_holdout
    .filter(pl.col("baseline_score") >= negative_holdout["baseline_score"].quantile(0.99))
     .sort("baseline_score", descending=True)
    .select("prompt", "adversarial", "baseline_score")
)

prompt,adversarial,baseline_score
str,bool,f32
"""Draft an edifying e-mail prais…",false,0.965381
"""Where can I find resources on …",false,0.951863
"""I just heard that Pfizer has r…",false,0.936285


## Распределение скоров на безопасных промптах

In [80]:
def plot_negative_score_distributions(
    data: pl.DataFrame,
    *,
    title='Score distributions on negative examples',
    n_bins=50,
):
    '''Plot scores from a table of already selected safe prompts.

    Required columns: baseline_score, classifier_score,
    finetuned_classifier_score. Displays and returns a Bokeh figure.
    '''
    output_notebook()

    negative_score_plot = figure(
        title=f'{title} (n={data.height})',
        x_axis_label='Positive-class score',
        y_axis_label='Fraction of negative examples',
        x_range=(0, 1),
        width=950,
        height=450,
        tools='pan,wheel_zoom,box_zoom,reset,save',
        tooltips=[('Score interval', '@left{0.00} – @right{0.00}'),
                  ('Fraction', '@top{0.0%}')],
    )
    negative_score_bins = np.linspace(0, 1, n_bins + 1)
    negative_score_columns = {
        'linear-probe': 'baseline_score',
        'ettin-classifer': 'classifier_score',
        'finetuned-ettin-classifer': 'finetuned_classifier_score',
    }

    for (model_name, score_column), color in zip(
        negative_score_columns.items(), Category10[3]
    ):
        scores = data[score_column].to_numpy()
        counts, edges = np.histogram(scores, bins=negative_score_bins)
        negative_score_plot.quad(
            top=counts / len(scores) if len(scores) else counts,
            bottom=0,
            left=edges[:-1],
            right=edges[1:],
            fill_color=color,
            fill_alpha=0.18,
            line_color=color,
            line_alpha=0.9,
            legend_label=model_names[model_name],
        )

    negative_score_plot.legend.location = 'top_right'
    negative_score_plot.legend.click_policy = 'hide'
    negative_score_plot.y_range.start = 0
    show(negative_score_plot)
    return negative_score_plot


negative_score_plot = plot_negative_score_distributions(
    negative_holdout,
    title='Score distributions on negative holdout examples',
)

Loading BokehJS ...

In [84]:
negative_score_plot = plot_negative_score_distributions(
    vanilla_negative_holdout,
    title='Score distributions on negative holdout examples',
)

Loading BokehJS ...

## Calibration curves

In [67]:
from sklearn.calibration import calibration_curve

output_notebook()

calibration_labels = holdout_data['label'].to_numpy()
calibration_plot = figure(
    title=f'Calibration curves on holdout (n={len(calibration_labels)})',
    x_axis_label='Mean predicted positive-class probability',
    y_axis_label='Observed fraction of positive examples',
    x_range=(0, 1),
    y_range=(0, 1),
    width=750,
    height=650,
    tools='pan,wheel_zoom,box_zoom,reset,save',
)
calibration_plot.line(
    [0, 1], [0, 1],
    line_color='gray',
    line_dash='dashed',
    line_width=2,
    legend_label='Perfect calibration',
)

for (model_name, display_name), color in zip(model_names.items(), Category10[3]):
    calibration_scores = np.asarray(holdout_predictions[model_name])
    if calibration_scores.ndim == 2:
        calibration_scores = calibration_scores[:, 1]
    fraction_positive, mean_predicted = calibration_curve(
        calibration_labels,
        calibration_scores,
        n_bins=10,
        strategy='uniform',
        pos_label=1,
    )
    calibration_plot.line(
        mean_predicted, fraction_positive,
        line_color=color,
        line_width=2,
        legend_label=display_name,
    )
    calibration_plot.scatter(
        mean_predicted, fraction_positive,
        color=color,
        size=8,
        legend_label=display_name,
    )

calibration_plot.legend.location = 'top_left'
calibration_plot.legend.click_policy = 'hide'
show(calibration_plot)

Loading BokehJS ...